# Ransomware Early Detection System
### Machine Learning + Anomaly Detection + Rule-Based Risk Scoring

**Purpose:** a defensive academic demonstration using synthetic behavior telemetry.  
**Safety:** this notebook never encrypts, renames, deletes, or modifies user files. It does not stop processes or isolate hosts. All events and data are simulated in memory.

**Important limitation:** the generated dataset is synthetic and intentionally simplified. Report the results as a proof-of-concept, not real-world ransomware detection accuracy.


In [ ]:
# CELL 1 — Install dependencies (Google Colab / Jupyter)
%pip -q install numpy pandas matplotlib seaborn scikit-learn joblib


In [ ]:
# CELL 2 — Imports and reproducibility
import json, random, hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
print("Imports successful.")


In [ ]:
# CELL 3 — Project directories (no existing folders are deleted)
LAB_DIR = Path("ransomware_detection_output")
LAB_DIR.mkdir(exist_ok=True)
REPORT_DIR = LAB_DIR / "reports"
MODEL_DIR = LAB_DIR / "models"
REPORT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
print("Output directory:", LAB_DIR.resolve())


In [ ]:
# CELL 4 — File entropy utility (read-only helper; optional demonstration)
def calculate_entropy(data: bytes) -> float:
    if not data:
        return 0.0
    counts = np.bincount(np.frombuffer(data, dtype=np.uint8), minlength=256)
    probs = counts[counts > 0] / len(data)
    return float(-(probs * np.log2(probs)).sum())

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

demo_bytes = b"Example text for a harmless entropy demonstration."
print("Example entropy:", round(calculate_entropy(demo_bytes), 3))
print("Example SHA-256:", sha256_bytes(demo_bytes))


In [ ]:
# CELL 5 — Define behavior features and safe synthetic data generator
FEATURES = [
    "files_modified", "files_renamed", "files_deleted", "directories_touched",
    "extensions_affected", "ransom_extensions", "events_per_second",
    "high_entropy_files", "entropy_average", "suspicious_process",
    "backup_activity", "network_connections"
]

def generate_sample(ransomware=False, rng=None):
    rng = rng or np.random.default_rng()
    if ransomware:
        modified = int(rng.integers(80, 1001))
        renamed = int(rng.integers(50, max(51, modified + 1)))
        return {
            "files_modified": modified, "files_renamed": renamed,
            "files_deleted": int(rng.integers(0, 101)),
            "directories_touched": int(rng.integers(5, 51)),
            "extensions_affected": int(rng.integers(3, 11)),
            "ransom_extensions": int(rng.integers(20, max(21, renamed + 1))),
            "events_per_second": int(rng.integers(30, 501)),
            "high_entropy_files": int(rng.integers(30, max(31, modified + 1))),
            "entropy_average": float(rng.uniform(7.0, 8.0)),
            "suspicious_process": 1, "backup_activity": int(rng.integers(0, 2)),
            "network_connections": int(rng.integers(5, 51)), "label": 1
        }
    modified = int(rng.integers(1, 31))
    return {
        "files_modified": modified,
        "files_renamed": int(rng.integers(0, min(5, modified) + 1)),
        "files_deleted": int(rng.integers(0, 4)),
        "directories_touched": int(rng.integers(1, 6)),
        "extensions_affected": int(rng.integers(1, 5)),
        "ransom_extensions": 0, "events_per_second": int(rng.integers(1, 11)),
        "high_entropy_files": int(rng.integers(0, 3)),
        "entropy_average": float(rng.uniform(3.0, 6.5)),
        "suspicious_process": int(rng.choice([0, 0, 0, 1])),
        "backup_activity": 0, "network_connections": int(rng.integers(0, 6)),
        "label": 0
    }

def build_dataset(n_per_class=1000, random_state=42):
    rng = np.random.default_rng(random_state)
    rows = [generate_sample(False, rng) for _ in range(n_per_class)]
    rows += [generate_sample(True, rng) for _ in range(n_per_class)]
    return pd.DataFrame(rows).sample(frac=1, random_state=random_state).reset_index(drop=True)

print("Feature count:", len(FEATURES))


In [ ]:
# CELL 6 — Generate and inspect dataset
df = build_dataset(n_per_class=1000, random_state=RANDOM_STATE)
print("Dataset shape:", df.shape)
display(df.head())
print("\\nClass counts (0=normal, 1=ransomware-like):")
display(df["label"].value_counts().sort_index())
print("\\nMissing values:")
display(df.isna().sum())


In [ ]:
# CELL 7 — Exploratory visualizations
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df, x="label", y="files_modified", ax=axes[0])
axes[0].set_title("File modifications by class")
sns.boxplot(data=df, x="label", y="entropy_average", ax=axes[1])
axes[1].set_title("Average entropy by class")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 8 — Train/test split
X = df[FEATURES].copy()
y = df["label"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))


In [ ]:
# CELL 9 — Train Random Forest classifier
rf_model = RandomForestClassifier(
    n_estimators=250, max_depth=12, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_prediction = rf_model.predict(X_test)
rf_probability = rf_model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, rf_prediction, zero_division=0))


In [ ]:
# CELL 10 — Evaluate model
metrics = {
    "accuracy": accuracy_score(y_test, rf_prediction),
    "precision": precision_score(y_test, rf_prediction, zero_division=0),
    "recall": recall_score(y_test, rf_prediction, zero_division=0),
    "f1": f1_score(y_test, rf_prediction, zero_division=0),
    "roc_auc": roc_auc_score(y_test, rf_probability)
}
display(pd.DataFrame([metrics]).round(4))


In [ ]:
# CELL 11 — Confusion matrix
cm = confusion_matrix(y_test, rf_prediction)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Ransomware-like"],
            yticklabels=["Normal", "Ransomware-like"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 12 — Feature importance
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)
display(importance)
plt.figure(figsize=(9, 5))
sns.barplot(data=importance, x="importance", y="feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 13 — Train Isolation Forest on normal training examples
isolation_model = IsolationForest(
    n_estimators=200, contamination=0.05, random_state=RANDOM_STATE
)
isolation_model.fit(X_train[y_train == 0])
print("Isolation Forest trained on normal-class training data.")


In [ ]:
# CELL 14 — Explainable rule scoring
def calculate_rule_score(features):
    score, reasons = 0, []
    rules = [
        (features["files_modified"] > 100, 20, "Rapid mass file modification"),
        (features["files_renamed"] > 50, 20, "Mass file renaming"),
        (features["entropy_average"] > 7.0, 20, "High average file entropy"),
        (features["suspicious_process"] == 1, 15, "Suspicious process indicator"),
        (features["backup_activity"] == 1, 15, "Backup activity indicator"),
        (features["network_connections"] > 10, 10, "Unusual network connection count"),
        (features["ransom_extensions"] > 0, 10, "Ransomware-like extension indicator")
    ]
    for matched, points, reason in rules:
        if matched:
            score += points
            reasons.append(reason)
    return min(score, 100), reasons


In [ ]:
# CELL 15 — Hybrid risk analysis
def analyze_behavior(feature_dict):
    row = pd.DataFrame([{name: feature_dict[name] for name in FEATURES}], columns=FEATURES)
    ml_score = float(rf_model.predict_proba(row)[0, 1] * 100)
    anomaly = int(isolation_model.predict(row)[0] == -1)
    anomaly_score = 20 if anomaly else 0
    rule, reasons = calculate_rule_score(feature_dict)
    final = round(min(100.0, rule * 0.50 + ml_score * 0.40 + anomaly_score * 0.10), 2)
    severity = ("NORMAL" if final <= 30 else "SUSPICIOUS" if final <= 60
                else "HIGH RISK" if final <= 80 else "CRITICAL")
    if anomaly:
        reasons.append("Behavior differs from the normal training profile")
    return {
        "rule_score": rule, "ml_score": round(ml_score, 2),
        "anomaly_score": anomaly_score, "final_risk_score": final,
        "severity": severity,
        "reasons": reasons or ["No configured warning indicators triggered"]
    }


In [ ]:
# CELL 16 — Test normal and ransomware-like simulated profiles
normal_test = generate_sample(False)
attack_test = generate_sample(True)
normal_result = analyze_behavior(normal_test)
attack_result = analyze_behavior(attack_test)
print("NORMAL PROFILE")
print(json.dumps(normal_result, indent=2))
print("\\nRANSOMWARE-LIKE PROFILE")
print(json.dumps(attack_result, indent=2))


In [ ]:
# CELL 17 — Early warning and safe response recommendations
def early_warning(result):
    score = result["final_risk_score"]
    if score > 80:
        alert = "CRITICAL: investigate immediately"
    elif score > 60:
        alert = "HIGH RISK: investigate"
    elif score > 30:
        alert = "SUSPICIOUS: monitor"
    else:
        alert = "NORMAL: no configured alert"
    return {**result, "alert": alert}

def response_recommendations(result):
    score = result["final_risk_score"]
    if score > 80:
        return ["Notify the security team", "Preserve relevant logs and evidence",
                "Follow the organization's approved incident-response plan",
                "Verify backup integrity using approved procedures"]
    if score > 60:
        return ["Escalate for analyst review", "Increase monitoring",
                "Preserve relevant logs"]
    if score > 30:
        return ["Record the event", "Continue monitoring"]
    return ["No additional action suggested by this demo"]

alert_result = early_warning(attack_result)
print(alert_result["alert"])
for item in response_recommendations(alert_result):
    print("-", item)


In [ ]:
# CELL 18 — Risk progression visualization (synthetic samples)
timeline = []
for index in range(15):
    sample = generate_sample(ransomware=(index >= 10))
    result = analyze_behavior(sample)
    timeline.append({"sample": index + 1, "risk_score": result["final_risk_score"]})
timeline_df = pd.DataFrame(timeline)
display(timeline_df)
plt.figure(figsize=(9, 4))
plt.plot(timeline_df["sample"], timeline_df["risk_score"], marker="o")
plt.axhline(60, linestyle="--", label="High-risk threshold")
plt.axhline(80, linestyle=":", label="Critical threshold")
plt.ylim(0, 105)
plt.xlabel("Synthetic sample sequence")
plt.ylabel("Risk score")
plt.title("Demonstration risk-score progression")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# CELL 19 — Create a forensic-style JSON report (simulation only)
report = {
    "incident_id": "SIM-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "detection_type": "Synthetic ransomware-like behavior analysis",
    "risk_result": alert_result,
    "sample_features": {k: (int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v, (np.floating,)) else v)
                        for k, v in attack_test.items() if k != "label"},
    "recommended_actions": response_recommendations(alert_result),
    "disclaimer": "Synthetic academic demonstration; not a real endpoint incident."
}
report_path = REPORT_DIR / "sample_forensic_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Report saved:", report_path.resolve())


In [ ]:
# CELL 20 — Save model and dataset
joblib.dump(
    {"random_forest": rf_model, "isolation_forest": isolation_model, "features": FEATURES},
    MODEL_DIR / "ransomware_detection_models.joblib"
)
df.to_csv(LAB_DIR / "synthetic_behavior_dataset.csv", index=False)
print("Saved model and synthetic dataset under:", LAB_DIR.resolve())


In [ ]:
# CELL 21 — Final project summary
print("RANSOMWARE EARLY DETECTION — PROJECT SUMMARY")
print("Dataset rows:", len(df))
print("Features:", len(FEATURES))
print("Accuracy on synthetic hold-out split:", round(metrics["accuracy"], 4))
print("Precision:", round(metrics["precision"], 4))
print("Recall:", round(metrics["recall"], 4))
print("F1:", round(metrics["f1"], 4))
print("ROC-AUC:", round(metrics["roc_auc"], 4))
print("Sample risk score:", alert_result["final_risk_score"], "/ 100")
print("Severity:", alert_result["severity"])
print("\\nNote: synthetic evaluation scores must not be described as real-world performance.")
